# Adaptive simulation with stabilizer branching

This notebook demonstrates simulation of an adaptive OpenQASM program with the QDK Clifford simulator. The program combines operations that are inexpensive for a stabilizer simulator with a small number of non-Clifford gates:

- a 14-qubit ring graph state within a 21-qubit register;
- three `t` gates and three inverse `tdg` gates;
- three teleportations with six mid-circuit measurements and feed-forward corrections;
- a bounded repeat-until-success loop; and
- configurable Pauli noise and qubit loss.

The T gates cause the stabilizer state to branch coherently. The echo formed by the inverse T gates and graph-state uncomputation makes the final result deterministic in a noiseless simulation, which gives us a straightforward correctness check.

In [ ]:
from collections import Counter

from qdk import Result, TargetProfile, openqasm
from qdk.simulation import NoiseConfig, run_qir

## Build the adaptive OpenQASM program

Each teleportation reports its two measurements before resetting and reusing the measured qubits. These results make the program's adaptive behavior visible and, in the noisy experiment below, expose loss of a qubit acted on by a T gate.

OpenQASM distinguishes a measurement `bit` from a classical `bool`. The RUS condition is therefore declared as a `bool`, allowing the measurement to control loop termination. The loop is also capped at eight attempts so every shot has a finite amount of work.

In [ ]:
qasm_source = r"""
OPENQASM 3.0;
include "stdgates.inc";

output bit[6] teleport_measurements;
output bool rus_success;
output int rus_attempts;
output bit[20] final_measurements;
qubit[21] q;

// Prepare a 14-qubit ring graph state.
h q[0]; h q[1]; h q[2]; h q[3]; h q[4]; h q[5]; h q[6];
h q[7]; h q[8]; h q[9]; h q[16]; h q[17]; h q[18]; h q[19];
cz q[0], q[1]; cz q[1], q[2]; cz q[2], q[3]; cz q[3], q[4];
cz q[4], q[5]; cz q[5], q[6]; cz q[6], q[7]; cz q[7], q[8];
cz q[8], q[9]; cz q[9], q[16]; cz q[16], q[17];
cz q[17], q[18]; cz q[18], q[19]; cz q[19], q[0];

// Introduce three non-Clifford phases.
t q[2]; t q[6]; t q[17];

// Teleport q[2] through q[10:11], then restore its logical location.
h q[10]; cx q[10], q[11]; cx q[2], q[10]; h q[2];
teleport_measurements[0] = measure q[2]; reset q[2];
teleport_measurements[1] = measure q[10]; reset q[10];
if (teleport_measurements[1]) { x q[11]; }
if (teleport_measurements[0]) { z q[11]; }
swap q[2], q[11];

// Teleport q[6] through q[12:13].
h q[12]; cx q[12], q[13]; cx q[6], q[12]; h q[6];
teleport_measurements[2] = measure q[6]; reset q[6];
teleport_measurements[3] = measure q[12]; reset q[12];
if (teleport_measurements[3]) { x q[13]; }
if (teleport_measurements[2]) { z q[13]; }
swap q[6], q[13];

// Teleport q[17] through q[14:15].
h q[14]; cx q[14], q[15]; cx q[17], q[14]; h q[17];
teleport_measurements[4] = measure q[17]; reset q[17];
teleport_measurements[5] = measure q[14]; reset q[14];
if (teleport_measurements[5]) { x q[15]; }
if (teleport_measurements[4]) { z q[15]; }
swap q[17], q[15];

// Echo the three T gates and unprepare the graph state.
tdg q[2]; tdg q[6]; tdg q[17];
cz q[19], q[0]; cz q[18], q[19]; cz q[17], q[18];
cz q[16], q[17]; cz q[9], q[16]; cz q[8], q[9];
cz q[7], q[8]; cz q[6], q[7]; cz q[5], q[6]; cz q[4], q[5];
cz q[3], q[4]; cz q[2], q[3]; cz q[1], q[2]; cz q[0], q[1];
h q[0]; h q[1]; h q[2]; h q[3]; h q[4]; h q[5]; h q[6];
h q[7]; h q[8]; h q[9]; h q[16]; h q[17]; h q[18]; h q[19];

// Succeed with probability 1/2 per attempt, up to eight attempts.
rus_success = false;
rus_attempts = 0;
while (!rus_success && rus_attempts < 8) {
    h q[20];
    rus_success = measure q[20];
    reset q[20];
    rus_attempts += 1;
}

final_measurements = measure q[0:19];
"""

## Compile to adaptive QIR

`TargetProfile.Adaptive` includes support for dynamic loop conditions. The narrower `Adaptive_RIF` profile supports measurement-conditioned gates, integers, and floating-point computations, but does not support terminating a loop from a measurement-derived Boolean.

In [ ]:
qir = openqasm.compile(
    qasm_source,
    output_semantics=openqasm.OutputSemantics.OpenQasm,
    target_profile=TargetProfile.Adaptive,
)
print(f"Compiled {len(str(qir)):,} characters of QIR")

## Run the noiseless circuit

The six teleportation results are random, but the feed-forward corrections preserve the state. The T/TDG echo then returns all 20 data and teleportation qubits to zero. The RUS attempt count should follow a geometric distribution with success probability 1/2, truncated after eight attempts.

In [ ]:
shots = 500
ideal_results = run_qir(qir, shots=shots, seed=42, type="clifford")

assert all(
    all(bit == Result.Zero for bit in final_measurements)
    for _, _, _, final_measurements in ideal_results
)

attempt_counts = Counter(attempts for _, _, attempts, _ in ideal_results)
successes = sum(success for _, success, _, _ in ideal_results)
print(f"RUS successes: {successes}/{shots}")
print("RUS attempt counts:", dict(sorted(attempt_counts.items())))
print("First teleportation records:")
for teleport_measurements, _, _, _ in ideal_results[:5]:
    print(" ", teleport_measurements)

## Add T-gate noise and loss

The noise model below applies a Z fault with probability 2% and qubit loss with probability 3% after each `t` gate. Noise is applied only to the three forward T gates so that loss is visible in the corresponding teleportation measurement. The simulator continues adaptive execution after either event, including reset and feed-forward operations.

In [ ]:
noise = NoiseConfig()
noise.t.z = 0.02
noise.t.loss = 0.03

noisy_results = run_qir(
    qir,
    shots=1000,
    noise=noise,
    seed=42,
    type="clifford",
)

loss_shots = sum(
    Result.Loss in teleport_measurements
    for teleport_measurements, _, _, _ in noisy_results
)
echo_failures = sum(
    any(bit != Result.Zero for bit in final_measurements)
    for _, _, _, final_measurements in noisy_results
)

print(f"Shots exposing T-gate loss: {loss_shots}/{len(noisy_results)}")
print(f"Shots with a nonzero or lost final qubit: {echo_failures}/{len(noisy_results)}")

## Scale to 111 qubits

A full-state simulation becomes impractical long before 111 qubits because it would need to store \(2^{111}\) complex amplitudes. Stabilizer branching instead allows the large Clifford portion of the program to retain tableau-like scaling while its additional cost is driven mainly by the small number of non-Clifford gates.

The generated program below expands the same experiment to:

- 100 data qubits in a ring graph state;
- five T-state teleportations using 10 ancillas;
- one RUS qubit;
- 111 qubits in total; and
- exactly 10 non-Clifford gates: five `t` and five `tdg` gates.

Every data qubit participates in the graph state, and the T gates are distributed around the ring. Python generates the repetitive OpenQASM statements so that the scaling example remains readable.

In [ ]:
def build_scaled_echo():
    data_qubits = 100
    targets = (9, 29, 49, 69, 89)
    ancilla_base = data_qubits
    rus_qubit = data_qubits + 2 * len(targets)
    total_qubits = rus_qubit + 1

    lines = [
        "OPENQASM 3.0;",
        'include "stdgates.inc";',
        "",
        f"output bit[{2 * len(targets)}] teleport_measurements;",
        "output bool rus_success;",
        "output int rus_attempts;",
        f"output bit[{total_qubits}] final_measurements;",
        f"qubit[{total_qubits}] q;",
        "",
    ]

    lines.extend(f"h q[{i}];" for i in range(data_qubits))
    lines.append("")
    lines.extend(f"cz q[{i}], q[{i + 1}];" for i in range(data_qubits - 1))
    lines.extend((f"cz q[{data_qubits - 1}], q[0];", ""))
    lines.extend(f"t q[{target}];" for target in targets)

    for index, target in enumerate(targets):
        bell = ancilla_base + 2 * index
        output = bell + 1
        first_result = 2 * index
        second_result = first_result + 1
        lines.extend(
            (
                "",
                f"h q[{bell}];",
                f"cx q[{bell}], q[{output}];",
                f"cx q[{target}], q[{bell}];",
                f"h q[{target}];",
                f"teleport_measurements[{first_result}] = measure q[{target}];",
                f"reset q[{target}];",
                f"teleport_measurements[{second_result}] = measure q[{bell}];",
                f"reset q[{bell}];",
                f"if (teleport_measurements[{second_result}]) {{ x q[{output}]; }}",
                f"if (teleport_measurements[{first_result}]) {{ z q[{output}]; }}",
                f"swap q[{target}], q[{output}];",
            )
        )

    lines.append("")
    lines.extend(f"tdg q[{target}];" for target in targets)
    lines.extend(("", f"cz q[{data_qubits - 1}], q[0];"))
    lines.extend(
        f"cz q[{i}], q[{i + 1}];" for i in reversed(range(data_qubits - 1))
    )
    lines.append("")
    lines.extend(f"h q[{i}];" for i in range(data_qubits))
    lines.extend(
        (
            "",
            "rus_success = false;",
            "rus_attempts = 0;",
            "while (!rus_success && rus_attempts < 8) {",
            f"    h q[{rus_qubit}];",
            f"    rus_success = measure q[{rus_qubit}];",
            f"    reset q[{rus_qubit}];",
            "    rus_attempts += 1;",
            "}",
            "",
            "final_measurements = measure q;",
        )
    )

    return "\n".join(lines), total_qubits, 2 * len(targets)


scaled_source, scaled_qubits, scaled_non_cliffords = build_scaled_echo()

In [ ]:
from time import perf_counter

compile_start = perf_counter()
scaled_qir = openqasm.compile(
    scaled_source,
    output_semantics=openqasm.OutputSemantics.OpenQasm,
    target_profile=TargetProfile.Adaptive,
)
compile_seconds = perf_counter() - compile_start

full_state_amplitudes = 1 << scaled_qubits
full_state_bytes = 16 * full_state_amplitudes
print(f"Qubits: {scaled_qubits}")
print(f"Non-Clifford gates: {scaled_non_cliffords}")
print(f"Full-state amplitudes: {full_state_amplitudes:.2e}")
print(f"Full-state storage at 16 bytes per amplitude: {full_state_bytes:.2e} bytes")
print(f"QIR size: {len(str(scaled_qir)):,} characters")
print(f"Compilation time: {compile_seconds:.3f} seconds")

In [ ]:
scaled_shots = 100
simulation_start = perf_counter()
scaled_results = run_qir(
    scaled_qir,
    shots=scaled_shots,
    seed=42,
    type="clifford",
)
simulation_seconds = perf_counter() - simulation_start

assert all(
    all(bit == Result.Zero for bit in final_measurements)
    for _, _, _, final_measurements in scaled_results
)

scaled_successes = sum(success for _, success, _, _ in scaled_results)
print(f"Verified all-zero echo results: {scaled_shots}/{scaled_shots}")
print(f"RUS successes: {scaled_successes}/{scaled_shots}")
print(f"Simulation time: {simulation_seconds:.3f} seconds")
print(f"Throughput: {scaled_shots / simulation_seconds:.1f} shots/second")

The noiseless checks exercise coherent stabilizer branching rather than converting the programs into Clifford-only approximations: the T phases remain coherent through teleportation and are cancelled only by the later inverse gates. The noisy run additionally exercises loss during a branched state, mid-circuit measurement of the lost qubit, reset, and continued adaptive execution. Finally, the 111-qubit run demonstrates how this approach reaches problem sizes far beyond full-state simulation when the number of non-Clifford gates remains modest.